In [1]:
from gradient_relevance_score import DistilBertAttributor

In [2]:
attributor = DistilBertAttributor(model_path="../final_distilbert-model/", max_length=256)

Using mps device


In [3]:
text = "Stephen J. Gordon (born 4 September 1986) is a chess grandmaster from Oldham, Greater Manchester, England. In September 2004 he took a break from his A-level studies at The Blue Coat School, Oldham to compete in the thirteenth Monarch Assurance Isle of Man International. In 2005, while still a FIDE Master, he finished 6th in the British Championships ahead of a Grandmaster and several International Masters. At the EU Individual Open Chess Championship held at Liverpool in 2006, he led the tournament after eight rounds and finished a very creditable (joint) second, a half point behind winner Nigel Short and level with Luke McShane among others. Probably his best result to date however, was second place in the 2007 British Championship, narrowly losing his share of the lead in the final round. In previous rounds, he defeated both tournament victor Jacob Aagaard and previous champion Jonathan Rowson. By 2008, his rating had reached grandmaster level, although the title itself had not yet been secured. At the British Championship in Liverpool, he almost repeated his performance of the previous year, by taking a share of third place. He was the British under-21 Champion each consecutive year between 2005 and 2008. He became a grandmaster on 1 August 2009. He has been one of the co-presenters of the chess podcast The Full English Breakfast since its inaugural show in October 2010."

In [4]:
target, important_tokens = attributor.compute_attributions(text, merge_scores=False)

In [5]:
def mark_important_tokens(_text, _important_tokens):
    marked_text = _text.replace("[","").replace("]","").lower()
    for token, _ in _important_tokens[:10]:
        marked_text = marked_text.replace(" {} ".format(token), " [{}] ".format(token))
    return marked_text

text_with_attributions = mark_important_tokens(text, important_tokens)

In [6]:
import requests, json

OPENAI_API_KEY = "EMPTY"
with open("../.openai_key.txt") as f:
    OPENAI_API_KEY = f.read().strip()
API_URL = "https://api.openai.com/v1/chat/completions"
MODEL = "gpt-3.5-turbo"  # Change to "gpt-4" if needed

HEADERS = {
    "Authorization": f"Bearer {OPENAI_API_KEY}",
    "Content-Type": "application/json"
}

def _send_request(messages, api=API_URL, headers=HEADERS, model=MODEL):
    payload = {
        "model": model,
        "messages": messages,
        "temperature": 0.0 
    }

    response = requests.post(api, headers=headers, json=payload)
    response.raise_for_status()
    data = response.json()
    reply = data["choices"][0]["message"]["content"]
    return reply

In [7]:
prompt = "Explain why the following text is {}anonymized. Words in square brackets [] are important words for the classification of this text. This is the text: ".format("not " if target == 0 else "")

def build_messages(prompt_appendix, text_):
    return [{"role": "user", "content": prompt_appendix + text_}]

In [8]:
_send_request(build_messages(prompt, text_with_attributions))

"The text is not anonymized because it includes the individual's full name, birthdate, place of birth, educational background, chess achievements, and specific dates and locations of tournaments. These details can easily identify the person as Stephen J. Gordon, a chess grandmaster from Oldham, Greater Manchester, England. Additionally, the text mentions specific chess tournaments, opponents, and achievements that are unique to Stephen J. Gordon, further confirming his identity."

In [9]:
from utils.read_jsonl import read_jsonl

eval_df = read_jsonl("../DB-bio/combined_val_and_val_sft_anonymized.jsonl")

In [10]:
eval_df = eval_df[eval_df["text"].apply(len) < 1100]
len(eval_df)

In [13]:
encoded = eval_df["text"].apply(lambda t: attributor.tokenizer.encode(t,truncation=True, max_length=256, padding="max_length"))
encoded = encoded.apply(lambda x: 1 if 0 in x else 0)
eval_df = eval_df[encoded == 1]
len(eval_df)

124

In [14]:
def get_natural_language_explanation(row):
    print(row.name)
    true_label = row["label"]
    _text = row["text"]
    _target, _important_tokens = attributor.compute_attributions(_text, merge_scores=False)
    _text = mark_important_tokens(_text, _important_tokens)
    _prompt = "Explain why the following text is classified as {}anonymized. Words in square brackets [] are important words for the classification of this text. This is the text: ".format("not " if _target == 0 else "")
    explanation = _send_request(build_messages(_prompt, _text))
    return explanation

In [15]:
len(eval_df)

124

In [16]:
eval_df["explanation"] = eval_df.apply(get_natural_language_explanation,axis=1)

2
8
12
14
18
19
29
32
33
34
45
49
53
55
56
60
62
63
70
73
75
76
83
86
87
88
91
92
95
98
100
115
117
122
128
132
135
137
139
140
143
146
154
156
157
161
174
175
181
182
183
188
189
194
195
205
208
209
223
227
233
234
235
242
248
249
251
254
257
262
269
270
272
275
276
287
288
292
295
296
298
299
303
306
313
314
316
318
327
329
331
334
338
341
343
349
358
359
360
365
368
371
380
389
392
397
399
400
418
429
431
432
434
437
438
451
452
466
468
476
477
478
484
485


In [17]:
eval_df.to_csv("../4_ExplanationResults/eval_df_with_explations_new.csv", index=False)

In [41]:
eval_df.head(1)["explanation"]

0    This text is classified as not anonymized beca...
Name: explanation, dtype: object

In [42]:
eval_df.tail(1)["explanation"]

485    This text is classified as anonymized because ...
Name: explanation, dtype: object